In [1]:
"""
Step 2: Clean dataset (verify images open correctly) and create
stratified train/val/test split (70/15/15) for the new Custom_data dataset.
"""
 
from pathlib import Path
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [2]:
# ---- CONFIG ----
CODE_DIR = Path(r"D:\B.Tech\CE-AI\Sem 7\MP\Code work")
 
df = pd.read_csv(CODE_DIR / "full_dataset.csv")
print(f"Loaded {len(df)} image records across {df['label'].nunique()} classes")

Loaded 13932 image records across 62 classes


In [3]:
# ---- Step 1: Verify each image opens correctly (drop corrupt files) ----
valid_rows = []
corrupt_files = []
 
for _, row in tqdm(df.iterrows(), total=len(df), desc="Verifying images"):
    try:
        with Image.open(row["filepath"]) as img:
            img.verify()  # checks file integrity without fully decoding
        valid_rows.append(row)
    except Exception as e:
        corrupt_files.append((row["filepath"], str(e)))
 
df_clean = pd.DataFrame(valid_rows).reset_index(drop=True)
 
print(f"\nValid images: {len(df_clean)}")
print(f"Corrupt/unreadable images dropped: {len(corrupt_files)}")
if corrupt_files:
    for path, err in corrupt_files[:10]:  # show first 10 only
        print(f"  - {path}: {err}")

Verifying images: 100%|██████████████████████████████████████████████████████████| 13932/13932 [04:05<00:00, 56.79it/s]



Valid images: 13932
Corrupt/unreadable images dropped: 0


In [4]:
# ---- Step 2: Re-check class counts after cleaning (in case corruption skewed a class) ----
class_counts = df_clean["label"].value_counts()
print(f"\nClasses after cleaning: {df_clean['label'].nunique()}")
print(f"Min images in a class: {class_counts.min()} | Max: {class_counts.max()}")


Classes after cleaning: 62
Min images in a class: 105 | Max: 627


In [5]:
# ---- Step 3: Stratified train/val/test split (70/15/15) ----
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    stratify=df_clean["label"],
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)
 
print(f"\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


Train: 9752 | Val: 2090 | Test: 2090


In [6]:
# ---- Step 4: Save CSVs ----
train_df.to_csv(CODE_DIR / "train_v2.csv", index=False)
val_df.to_csv(CODE_DIR / "val_v2.csv", index=False)
test_df.to_csv(CODE_DIR / "test_v2.csv", index=False)
 
print(f"\nSaved train_v2.csv, val_v2.csv, test_v2.csv to: {CODE_DIR}")


Saved train_v2.csv, val_v2.csv, test_v2.csv to: D:\B.Tech\CE-AI\Sem 7\MP\Code work
